# Intercoder reliability

Computes agreement statistics for the manual annotation process without displaying clinical text or record identifiers.


In [ ]:
# Repository-local path configuration
from pathlib import Path
import os

PROJECT_ROOT = Path(os.environ.get("GEP_PROJECT_ROOT", Path.cwd())).resolve()
DATA_DIR = Path(os.environ.get("GEP_DATA_DIR", PROJECT_ROOT / "data")).resolve()
MODEL_DIR = Path(os.environ.get("GEP_MODEL_DIR", PROJECT_ROOT / "models")).resolve()
RESULTS_DIR = Path(os.environ.get("GEP_RESULTS_DIR", PROJECT_ROOT / "results")).resolve()
FIGURES_DIR = Path(os.environ.get("GEP_FIGURES_DIR", PROJECT_ROOT / "figures")).resolve()

for directory in (MODEL_DIR, RESULTS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score


## Load annotation labels


In [ ]:
annotations = pd.read_csv(DATA_DIR / 'annotations.csv')
training = pd.read_csv(DATA_DIR / 'GEP_train_80_20.csv')
testing = pd.read_csv(DATA_DIR / 'GEP_test_80_20.csv')
reference = pd.concat([training, testing], ignore_index=True)
reliability = annotations.merge(
    reference[['note_id', 'label']],
    left_on='encounter_note_id', right_on='note_id', how='inner',
    suffixes=('_coder', '_reference'),
)
print(f'Matched annotation records: {len(reliability)}')


## Calculate Cohen kappa and percentage agreement


In [ ]:
coder_labels = reliability['label_coder'].apply(
    lambda value: 0 if value in {'exclude', 'positive', 'neutral'} else 1
).astype(int).to_numpy()

reference_labels = np.array([
    1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1,
    1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0,
], dtype=int)

if len(coder_labels) != len(reference_labels):
    raise ValueError(
        f'Expected {len(reference_labels)} reliability records; found {len(coder_labels)}.'
    )

kappa = cohen_kappa_score(coder_labels, reference_labels)
agreement = float((coder_labels == reference_labels).mean())
print(f"Cohen kappa: {kappa:.3f}")
print(f"Percentage agreement: {agreement * 100:.1f}%")
